In [0]:
CATALOG = "medallion_catalog" 
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{SILVER_SCHEMA}.stg_orders

WITH bronze_source AS (
    SELECT *
    FROM {CATALOG}.{BRONZE_SCHEMA}.orders
)

SELECT  order_id
        , customer_id
        , cast(campaign_id as bigint) as campaign_id
        , created_at
        , lower(status) as order_status
        , lower(sales_channel) as sales_channel
        , payment_method
        , coupon_code
        , lower(shipping_method) as shipping_method
        , shipping_fee
        , tax_amount
        , order_total
        , ship_city
        , ship_state
        , case
            when ship_country = "US" then "United States"
            when ship_country = "UK" and ship_state = "ENG" then "England"
            when ship_country = "UK" and ship_state = "SCT" then "Scotland"
            else ship_country
          end as ship_country
        , shipped_at
        , delivered_at
FROM bronze_source

"""

)

In [0]:
spark.sql(f"""
    COMMENT ON TABLE {CATALOG}.{SILVER_SCHEMA}.stg_orders IS
    "This table is a staging table for the orders table. It is used to transform the data from the bronze layer to the silver layer."
"""
)

description = {
    "order_id": "Unique identifier for the order",
    "customer_id": "Unique identifier for the customer",
    "campaign_id": "Unique identifier for the campaign",
    "created_at": "Timestamp when the order was created",
    "order_status": "Status of the order",
    "sales_channel": "Channel through which the order was placed",
    "payment_method": "Method of payment used for the order",
    "coupon_code": "Code used for the coupon",
    "shipping_method": "Method of shipping used for the order",
    "shipping_fee": "Shipping fee for the order",
    "tax_amount": "Tax amount for the order",
    "order_total": "Total amount of the order",
    "ship_city": "City where the order was shipped",
    "ship_state": "State where the order was shipped",
    "ship_country": "Country where the order was shipped",
    "shipped_at": "Timestamp when the order was shipped",
    "delivered_at": "Timestamp when the order was delivered"
}

for col, desc in description.items():
    spark.sql(f"ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.stg_orders ALTER COLUMN {col} COMMENT '{desc}'")

spark.sql(f"ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.stg_orders ALTER COLUMN order_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.stg_orders ADD CONSTRAINT order_pk PRIMARY KEY (order_id)")